<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Appendix A: *Constructing a Mesh Network*

## Metadata

---
### Contents  
> *Sample Grid Construction*\
> *Regional*\
> *Elevation*\
> *Social*\
> *Infrastructure*\
> *Fuel Sources*\
> *Housekeeping and Export*\
---
### Notes

- ArcGIS workflow for constructing a mesh network of sampling points equally spaced throughout California.
---
### Inputs
- `ops.csv` - A csv containing all operations performed in ArcGIS
- `lay.csv` - A csv containing all layers used and created in ArcGIS
- `Sampling_grids.csv` - The grid data
---
### Outputs  
- `cleaned_grids.csv` - Grdi dataset prepared for merging
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)

## Load Files

In [4]:
operations = pd.read_csv('../data/raw/ops.csv')
operations = operations.fillna('')
layers = pd.read_csv('../data/raw/lay.csv')
meta = pd.read_csv('../data/raw/Sampling_grids_meta.csv')
grids_raw = pd.read_csv('../data/raw/Sampling_grids.csv')

## Constructing the sampling Grid

<img src="../data/maps/grids.png" width="400">

## Sampling Grid Operations

### Data Sources: 
- California shapefile: <https://data.ca.gov/dataset/ca-geographic-boundaries>

In [5]:
operations[operations['executive_operation']=='Sampling Grid']

,executive_operation,operation_id,operation_title,input_names,tool_used,operation_settings,output_names,operation_notes
0,Sampling Grid,1,Reproject California shapefile,['CA_Counties'],Data Management Tools -> Projections and Transformations -> Project,,['CA_Counties'],
1,Sampling Grid,2,Dissolve california to state boundary,['CA_Counties'],Data Management Tools -> Generalization -> Dissolve,,['California_state'],
2,Sampling Grid,3,Create mesh network,['California_state'],Data Management Tools -> Sampling -> Create Fishnet,Fishnet origination: X -380101.0657 Y -605327.0481; Cell size width = 46000; Cell size height = 46000,['Sampling_grid_unclipped'],Creates a grid of sampling points equally space 46 Km apart throught the entire state of California. Output must be clipped to California boundaries.
3,Sampling Grid,4,Select and export all grids that intersect with California shape file into a feature layer,"['California_state', 'Sampling_grid_unclipped']",na -> na -> Select by Location,Relationship: Intersect,['Sampling_grid'],This operation creates a new layer by selecting all grids with centroids i
4,Sampling Grid,5,"Calculate the centroid geometry (northing,easting) and created fields in sampling_grid",['Sampling_grid'],Data Management Tools -> Features -> Calculate Geometry Attributes,Relationship: Within,['Sampling_grid'],These centroids are used to extract latitude and longitude for modelling
5,Sampling Grid,6,Create a new layer that contains only the portions of California that fall within the grids,"['California_state', 'Sampling_grid']",Analysis Tools -> Overlay -> Intersect,,['Sampling_grid_in_cali'],Temporary layer to calculate area of california within each grid
6,Sampling Grid,7,Calculate area in each grid that falls inside California bloundaries,['Sampling_grid_in_cali'],na -> na -> Calculate Geometry,Area (geodesic),['Sampling_grid_in_cali'],Units: Square meters
7,Sampling Grid,8,Join area in California field back to sampling_grid layer,['Sampling_grid_in_cali'],Analysis Tools -> Overlay -> Spatial Join,Field map: area_in_cali = sum of calculated areas within each grid; Match Option: Contains,['Sampling_grid_area'],Remove unneccessary features from shapefile layer
35,Sampling Grid,34,map weather sampling points,['samples_daily.csv'],Create Points From Table -> na -> XY Table to Point,,['samples_daily_map'],
36,Sampling Grid,35,Project weather samples to project crs,['samples_daily_map'],Data Management Tools -> Projections and Transformations -> Project,,['samples_daily_project'],


## Region Data Operations

### Data Sources:

- Wildland Urban Interface <https://gis.data.ca.gov/datasets/CALFIRE-Forestry::wildland-urban-interface/explore?location=34.403601%2C-118.894358%2C9.95>
- California Eco Regions: <https://data.fs.usda.gov/geodata/edw/datasets.php?xmlKeyword=calveg>


In [6]:
operations[operations['executive_operation']=='Region']

,executive_operation,operation_id,operation_title,input_names,tool_used,operation_settings,output_names,operation_notes
8,Region,9,Summarize WUI zones,"['Sampling_grid_area', 'California_state']",Analysis Tools -> Statistics -> Summarize Within,,"['Sampling_grid_WUI_zones', 'WUI_DESC_Summary']",Summary Fields: Area - Sum; Group Field: WUI_DESC
9,Region,10,Pivot WUI area summary table,['WUI_DESC_Summary'],Data Management Tools -> Table -> Pivot Table,Input Table: WUI_Desc_Summary; Input Field: Grid_ID; Pivot Field: WUI_type; Value Field: Sum,['WUI_sum'],Table to join to grid layer
10,Region,11,Join summary table to sampling grid,"['Sampling_grid_area', 'WUI_sum']",Joins and Relates -> na -> Add Join,,['Sampling_grid_WUI'],Rename fields to avoid confusion; Export features after join
14,Region,15,Find most dominant province in each grid,"['Sampling_grid_WUI', 'ecoregions']",Analysis Tools -> Statistics -> Summarize Within,Group Field: Province Area; check minority and majority attributes; check add group percentages,['Sampling_grid_Province'],
15,Region,16,Find most dominant eco section in each grid,"['Sampling_grid_Province', 'ecoregions']",Analysis Tools -> Statistics -> Summarize Within,Group Field: Section Area; check minority and majority attributes; check add group percentages,['Sampling_grid_Section'],
16,Region,17,Decode domain names with lookup tables,"['Sampling_grid_Province', 'Sampling_grid_Section']",Joins and Relates -> na -> Add Join,,['Sampling_grid_eco'],


## Elevation Data Operations

### Data Sources:

- Elevation Rasters <https://apps.nationalmap.gov/downloader/>

In [7]:
operations[operations['executive_operation']=='Elevation']

,executive_operation,operation_id,operation_title,input_names,tool_used,operation_settings,output_names,operation_notes
11,Elevation,12,Create a mosaic raster,['Multiple_DEM_cali_rasters'],Data Management Tools -> Raster Dataset -> Mosaic to New Raster,"Inputs: all .tif files downloaded from USGS explorer, see appendix for list; Pixel Type: 32 bit float; Cell Size = .001",['California_DEM_mosaic'],Performed in a fresh ArcGIS environment and final raster imported into project
12,Elevation,13,Reproject mosaic raster,['California_DEM_mosaic'],Data Management Tools -> Projections and Transformations -> Project Raster,,['California_DEM_mosaic_projected'],Completed in a fresh ArcGIS instance
13,Elevation,14,Reproject California ECO regions layer,['EV_CalvegZones_Ecoregions'],Data Management Tools -> Projections and Transformations -> Project,,['ecoregions'],
21,Elevation,22,Clip Raster to California Boundaries,['California_DEM_mosaic_projected'],Data Management Tools -> Raster Processing -> Clip Raster,,['DEM_California_Clip'],
22,Elevation,23,Calculate slope,['DEM_California_Clip'],Analyst Tools -> Surface -> Slope,,['Slope_California'],
23,Elevation,24,Calculate aspect,['DEM_California_Clip'],Analyst Tools -> Surface -> Aspect,,['Aspect_California'],
24,Elevation,25,Calculate elevation stats,['DEM_California_Clip'],Spatial Analyst -> Zonal -> Zonal Statistics,,['Elevation_zonal_stats'],
25,Elevation,26,Join elevation stats to table,"['DEM_California_Clip', 'Elevation_zonal_stats']",Joins and Relates -> na -> Add Join,,['sampling_grid_elevation'],"Keep mean, range, and standard deviation"
26,Elevation,25,Calculate slope stats,['Slope_California'],Spatial Analyst -> Zonal -> Zonal Statistics,,['Slope_zonal_stats'],
27,Elevation,26,Join slope stats to table,['Slope_zonal_stats'],Joins and Relates -> na -> Add Join,,['sampling_grid_slope'],"Keep mean, range, and standard deviation"


## Social Data Operations

### Data Sources:
- Census Tract Data: <https://catalog.data.gov/dataset/tiger-line-shapefile-2021-state-california-census-tracts>
- Census Block Data: <https://catalog.data.gov/dataset/tiger-line-shapefile-current-state-california-2020-census-block>
- ACS Income Census Data: <https://data.census.gov/table/ACSDP1Y2017.DP05?q=United+States&table=DP05&g=010XX00US&lastDisplayedRow=29&vintage=2017&layer=state&cid=DP05_0001E&tid=ACSDP1Y2017.DP05>

In [8]:
operations[operations['executive_operation']=='Social']

,executive_operation,operation_id,operation_title,input_names,tool_used,operation_settings,output_names,operation_notes
32,Social,31,Join medain income stats to census tract geometry table,['tl_2021_06_tract.shp'],Joins and Relates -> na -> Add Join,,['median_income_tract'],
33,Social,32,Project income data to project crs,['median_income_tract'],Data Management Tools -> Projections and Transformations -> Project,,['median_income_tract_project'],
34,Social,33,Take the average median income of all tracts in smapling grid,['median_income_tract_project'],Analysis Tools -> Statistics -> Summarize Within,,['Sampling_grid_income'],
45,Social,44,Project census block data to project crs,['tl_2024_06_tabblock20.shp'],Data Management Tools -> Projections and Transformations -> Project,,['census_blocks_project'],
46,Social,45,Summarize population data within each grid,['census_blocks_project'],Analysis Tools -> Statistics -> Summarize Within,,['Sampling_grid_block'],


## Infrastructure Data Operations

### Data Sources:
- Transmission Lines:  <https://cecgis-caenergy.opendata.arcgis.com/datasets/CAEnergy::california-electric-transmission-lines-1/explore>
- California All Public Roads: <https://gisdata-caltrans.opendata.arcgis.com/datasets/2d56e65de89c418780056651640291e8_0/about>

In [9]:
operations[operations['executive_operation']=='Infrastructure']

,executive_operation,operation_id,operation_title,input_names,tool_used,operation_settings,output_names,operation_notes
39,Infrastructure,38,Project power lines layer to project crs,['TransmissionLine_CEC.shp'],Data Management Tools -> Projections and Transformations -> Project,,['Sampling_grid_power'],
40,Infrastructure,39,Count number and length of power lines in each grid,['TransmissionLine_CEC.shp'],Analysis Tools -> Statistics -> Summarize Within,,['Sampling_grid_power'],Add number of lines and total length of lines fields to fields in grid dataset
41,Infrastructure,40,Project roads layer to project crs,['California_All_Public_Roads_Network.shp'],Data Management Tools -> Projections and Transformations -> Project,,['all_roads_project'],Add number of lines and total length of lines fields to fields in grid dataset
42,Infrastructure,41,Calculate length of all road polyline semgents,['all_roads_project'],Add Field -> na -> Calculate Geometry,Units: Meters,['all_roads_project'],
43,Infrastructure,42,Count total length of road present in each grid,['all_roads_project'],Analysis Tools -> Statistics -> Summarize Within,,['Sampling_grid_road'],
44,Infrastructure,43,Add road density field,['Sampling_grid_road'],Add Field -> na -> Calculate Field,road_density = length_of_roads / area_in_cali * 100,['Sampling_grid_road'],


## Fuel Sources Data Operations

### Data Sources:
- Land Cover: <https://data.ca.gov/dataset/nlcd-2021-land-cover-california-subset1>

In [10]:
operations[operations['executive_operation']=='Fuel']

,executive_operation,operation_id,operation_title,input_names,tool_used,operation_settings,output_names,operation_notes
17,Fuel,18,Calculate area of each land cover class in each grid,"['Sampling_grid_eco', 'nlcd_2001_land_cover_ca_20210604']",Spatial Analyst Tools -> Zonal -> Tabulate Area,Class Field: NLCD_Name,['Land_cover_area'],
18,Fuel,19,Group land cover classes,"['Land_cover_area', 'land_cover_lookup']",na -> na -> Calculate Field,Add new fields for each larger group and assign each subgroup by summing together,['Land_cover_area'],
19,Fuel,20,Join groups to grids,"['Sampling_grid_eco', 'Land_cover_area']",Joins and Relates -> na -> Add Join,,['Sampling_grid_nlcd_area'],
20,Fuel,21,Convert areas to percentages of California Area,['Sampling_grid_nlcd_area'],na -> na -> Calculate Field,,['Sampling_grid_nlcd_area'],5 fields produced


## Housekeeping and Export

In [11]:
drop = ['grid_area','geometry','Shape_Length','Shape_Area']
grids = grids_raw.drop(columns=drop)
grids = grids.rename(columns={'Influence_Zone':'influence_zone', 'Total_Population':'total_population'})

In [12]:
basic_explore(grids)

Rows:  236  Columns:  41
Duplicates  0
Total NA values:  126  of  9676 datapoints


In [13]:
grids.isna().sum()

grid_id                           0
centroid_easting                  0
centroid_northing                 0
influence_zone                   32
interface_zone                   52
intermix_zone                    42
dominant_province_description     0
dominant_province_percent         0
sum_province_area                 0
dominant_section_description      0
sum_section_area                  0
dominant_section_percent          0
area_in_cali                      0
forest_percent                    0
developed_percent                 0
other_percent                     0
shrub_grass_percent               0
wetlands_percent                  0
elevation_range                   0
elevation_mean                    0
elevation_std                     0
slope_max                         0
slope_range                       0
slope_mean                        0
slope_std                         0
northness_mean                    0
eastness_mean                     0
median_income               

**Note** The three fields with NA values are grids which contain none of the zones. Set these values to 0.

In [14]:
grids = grids.fillna(0)

In [15]:
grids.to_csv('../data/processed/cleaned_grids.csv',index = False)
print("All datasets saved successfully.")

All datasets saved successfully.
